In [1]:
import os
import io
import json
import time
import threading
from datetime import datetime
from pathlib import Path
from typing import List, Optional

import cv2
import numpy as np
import pandas as pd
import requests
from PIL import Image
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from google import genai
from google.genai import types

from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "🚨 GEMINI_API_KEY를 찾을 수 없습니다. "
        ".env 파일에 GEMINI_API_KEY=본인키 형식으로 저장했는지 확인해주세요."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini client 연결 완료")

✅ Gemini client 연결 완료


In [4]:
PROJECT_ROOT = Path.cwd()

# 입력 CSV 경로
INPUT_CSV_PATH = PROJECT_ROOT / "data" / "input" / "shorts-analysis_test-1 copy.csv"

# 출력 폴더
OUTPUT_DIR = PROJECT_ROOT / "data" / "thumbnails"
RAW_DIR = OUTPUT_DIR / "raw"
JSON_DIR = OUTPUT_DIR / "analysis_json"

CSV_OUTPUT_PATH = OUTPUT_DIR / "thumbnail_features.csv"
PARQUET_OUTPUT_PATH = OUTPUT_DIR / "thumbnail_features.parquet"
LOG_PATH = OUTPUT_DIR / "processing_log.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

# Gemini 모델
MODEL_NAME = "gemini-2.5-flash"

# 버전 관리
PROMPT_VERSION = "v1"
SCHEMA_VERSION = "v1"

# 재시도 설정
GEMINI_MAX_RETRIES = 5

# process_one_video 전체 재시도는 Gemini 내부 재시도와 중복되므로 1 추천
PROCESS_MAX_RETRIES = 1

# 병렬 처리 기본값
DEFAULT_MAX_WORKERS = 2
DEFAULT_CHUNK_SIZE = 10
DEFAULT_CHUNK_SLEEP_SECONDS = 3

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_CSV_PATH:", INPUT_CSV_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image
INPUT_CSV_PATH: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image\data\input\shorts-analysis_test-1 copy.csv
OUTPUT_DIR: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image\data\thumbnails


In [ ]:
class ThumbnailGeminiResult(BaseModel):
    visual_summary: str = Field(description="썸네일 전체 장면을 한국어 1문장으로 요약")
    main_objects: List[str] = Field(description="주요 객체 리스트. 예: person, product, text, brand_logo")

    thumbnail_category: str = Field(
        description="정보전달형, 제품홍보형, 브랜드이미지형, 이벤트홍보형, 인터뷰/인물형, 예능/콘텐츠형, 리뷰/비교형, 기타 중 하나"
    )

    has_person: bool
    person_count: int
    has_face: bool
    facial_expression: str = Field(description="smile, neutral, serious, surprised, unclear, none 등")

    has_text: bool
    text_on_thumbnail: str = Field(description="이미지 안에 보이는 문구. 없으면 빈 문자열")
    text_language: str = Field(description="ko, en, mixed, none, unknown 등")
    text_size_level: str = Field(description="none, small, medium, large 중 하나")

    brand_logo_visible: bool
    brand_name_visible: bool
    brand_name_detected: str = Field(description="감지된 브랜드명. 불확실하면 빈 문자열")

    dominant_colors: List[str] = Field(description="주요 색상 리스트. 예: blue, white, black")
    color_tone: str = Field(description="warm, cool, neutral 중 하나")
    background_complexity: str = Field(description="low, medium, high 중 하나")

    composition_style: str = Field(
        description="person_centered, product_centered, text_centered, mixed, other 중 하나"
    )
    attention_hook: str = Field(description="클릭을 유도하는 핵심 시각 요소를 한국어 1문장으로 설명")

    clickbait_level: int = Field(description="자극성/낚시성 정도. 0~5 정수")
    professional_level: int = Field(description="기업 유튜브 썸네일로서의 완성도. 0~5 정수")

    target_audience_guess: str = Field(description="예상 타겟 시청자. 예: general_consumers, job_seekers, investors 등")

In [ ]:
THUMBNAIL_PROMPT = """
너는 기업 유튜브 썸네일 분석 전문가다.
입력된 유튜브 썸네일 이미지를 보고 기업 유튜브 관점에서 시각적 특징을 분석하라.

분석 원칙:
1. 기업 공식 채널 또는 브랜드 채널의 썸네일이라고 가정하고 평가한다.
2. 썸네일의 인물, 제품, 텍스트, 브랜드 로고, 색감, 구도, 클릭 유도 요소를 분석한다.
3. 불확실한 내용은 과도하게 단정하지 않는다.
4. clickbait_level은 자극성/낚시성 정도를 0~5 정수로 평가한다.
   - 0: 전혀 자극적이지 않음
   - 1: 매우 약함
   - 2: 약간 있음
   - 3: 보통
   - 4: 강함
   - 5: 매우 강함
5. professional_level은 기업 유튜브 썸네일로서의 완성도를 0~5 정수로 평가한다.
   - 0: 매우 낮음
   - 1: 낮음
   - 2: 다소 낮음
   - 3: 보통
   - 4: 높음
   - 5: 매우 높음
6. thumbnail_category는 반드시 아래 중 하나로 분류한다.
   - 정보전달형
   - 제품홍보형
   - 브랜드이미지형
   - 이벤트홍보형
   - 인터뷰/인물형
   - 예능/콘텐츠형
   - 리뷰/비교형
   - 기타
7. composition_style은 반드시 아래 중 하나로 분류한다.
   - person_centered
   - product_centered
   - text_centered
   - mixed
   - other
8. background_complexity는 low / medium / high 중 하나로 분류한다.
9. text_size_level은 none / small / medium / large 중 하나로 분류한다.
10. color_tone은 warm / cool / neutral 중 하나로 분류한다.
"""

In [7]:
def load_input_csv(input_csv_path: Path) -> pd.DataFrame:
    if not input_csv_path.exists():
        raise FileNotFoundError(f"입력 CSV 파일이 없습니다: {input_csv_path}")

    df = pd.read_csv(input_csv_path)

    if "video_id" not in df.columns:
        raise ValueError("입력 CSV에 video_id 컬럼이 필요합니다.")

    df["video_id"] = df["video_id"].astype(str).str.strip()
    df = df[df["video_id"].notna()]
    df = df[df["video_id"] != ""]
    df = df.drop_duplicates(subset=["video_id"])

    return df

In [8]:
# CSV 로드 함수 테스트
input_df = load_input_csv(INPUT_CSV_PATH)

print("입력 데이터 크기:", input_df.shape)
display(input_df.head())

입력 데이터 크기: (6, 26)


,Unnamed: 0,video_id,title,channel_id,channel_title,description,upload_date,tags,category_id,view_count,...,has_paid_product_placement,thumbnail,thumbnail_h,thumbnail_w,caption,year,verdict,reasoning,final_url,domain
0,1630,knCQBlBKSRM,[B 다이렉트샵] 필사즉생의 각오로 임하는 발음 연습,UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,죽고자 하면 산다고 하잖아요\n그래서 죽기살기로 연습해달라 했죠.\n그렇게 하다보니...,2026-02-13 04:20:50+00:00,NaN,1,3615612,...,False,https://i.ytimg.com/vi/knCQBlBKSRM/hqdefault.jpg,360,480,False,2026,shorts,kept /shorts route,https://www.youtube.com/shorts/knCQBlBKSRM,IT
1,1631,NCVxpXxTMSU,[B 다이렉트샵] 고음불가? NO! 발음불가? YES,UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,연습 50일 째. 광고주 담당자 얼굴에서\n안색이라는 것이 점차 사라져가고 있습니다...,2026-02-13 04:20:46+00:00,NaN,1,3041416,...,False,https://i.ytimg.com/vi/NCVxpXxTMSU/hqdefault.jpg,360,480,False,2026,shorts,kept /shorts route,https://www.youtube.com/shorts/NCVxpXxTMSU,IT
2,1640,LRmLOsmMHts,에이닷과 대화하며 즐기는 B tv IPTV로 바꾸자! (문의 1600-2345),UCwR9k7QggFEfeHkvTR31JcA,SK브로드밴드_B tv,아직도 IPTV로 안 바꾸셨어요?\n\n주인공이 뛰는 미국 영화가 뭐였더라?\n궁금...,2026-01-08 07:58:00+00:00,NaN,1,261176,...,False,https://i.ytimg.com/vi/LRmLOsmMHts/hqdefault.jpg,360,480,False,2026,shorts,kept /shorts route,https://www.youtube.com/shorts/LRmLOsmMHts,IT
3,655,twi9zUxsXu0,🍓 + 🍫 +🌽 + 🥘 = 😋 | CU X 프롬서희 이모티콘 먹방 2월 3주차 #...,UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],이렇게 귀엽기 잇씨유? (✿◡‿◡)\n\n귀여운 꽃카 콜라보 신상과\n한국인은 못참...,2023-02-15 09:00:18+00:00,"먹방, MUKBANG, mukbang, 이모지먹방, 이모티콘먹방, emoji, em...",24,189406,...,False,https://i.ytimg.com/vi/twi9zUxsXu0/hqdefault.jpg,360,480,False,2023,shorts,kept /shorts route,https://www.youtube.com/shorts/twi9zUxsXu0,FnB
4,656,Xfu1kCID0Ls,[연예뉴스] 두 아티스트의 불화설이 제기돼... | CU콘서트 #4강 #B조 #3라운드,UCsM07dUwo0WOWhFbVsG8K6A,CU [씨유튜브],스탠딩 코미디의 부활! [CU콘서트]\n\nCU 공채 개그맨에 한 걸음 더 다가선\...,2023-02-13 09:00:40+00:00,"CU편의점, CU, 씨유, 헤이루, Heyroo, cu콘서트, cuconcert, ...",24,46925,...,False,https://i.ytimg.com/vi/Xfu1kCID0Ls/hqdefault.jpg,360,480,False,2023,shorts,kept /shorts route,https://www.youtube.com/shorts/Xfu1kCID0Ls,FnB


In [9]:
def build_thumbnail_url(video_id: str) -> str:
    return f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg"


def download_thumbnail(video_id: str, thumbnail_url: str) -> str:
    save_path = RAW_DIR / f"{video_id}.jpg"

    # 이미 다운로드한 파일이 있으면 재사용
    if save_path.exists() and save_path.stat().st_size > 1000:
        return str(save_path)

    response = requests.get(thumbnail_url, timeout=15)
    response.raise_for_status()

    if len(response.content) < 1000:
        raise ValueError(f"다운로드된 이미지 용량이 너무 작습니다: {video_id}")

    with open(save_path, "wb") as f:
        f.write(response.content)

    return str(save_path)

In [ ]:
def extract_opencv_features(image_path: str) -> dict:
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"이미지 파일이 없습니다: {image_path}")

    if image_path.stat().st_size < 1000:
        raise ValueError(f"이미지 파일 크기가 너무 작습니다: {image_path}")

    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"OpenCV가 이미지를 읽지 못했습니다: {image_path}")

    height, width = image.shape[:2]

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    brightness_mean = float(np.mean(hsv[:, :, 2]))
    saturation_mean = float(np.mean(hsv[:, :, 1]))
    contrast_std = float(np.std(gray))

    aspect_ratio_value = width / height if height else None

    if aspect_ratio_value is not None:
        if abs(aspect_ratio_value - 16 / 9) < 0.05:
            aspect_ratio = "16:9"
        elif abs(aspect_ratio_value - 4 / 3) < 0.05:
            aspect_ratio = "4:3"
        elif abs(aspect_ratio_value - 1) < 0.05:
            aspect_ratio = "1:1"
        elif aspect_ratio_value < 1:
            aspect_ratio = "vertical"
        else:
            aspect_ratio = "other"
    else:
        aspect_ratio = "unknown"

    return {
        "image_width": int(width),
        "image_height": int(height),
        "aspect_ratio": aspect_ratio,
        "brightness_mean": round(brightness_mean, 2),
        "contrast_std": round(contrast_std, 2),
        "saturation_mean": round(saturation_mean, 2),
    }

In [11]:
def analyze_thumbnail_with_gemini(
    image_path: str,
    max_retries: int = GEMINI_MAX_RETRIES,
) -> dict:
    with open(image_path, "rb") as f:
        image_bytes = f.read()

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(
                        data=image_bytes,
                        mime_type="image/jpeg",
                    ),
                    THUMBNAIL_PROMPT,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    response_mime_type="application/json",
                    response_schema=ThumbnailGeminiResult,
                ),
            )

            return json.loads(response.text)

        except Exception as e:
            last_error = e

            print(f"⚠️ Gemini 호출 실패 attempt={attempt}/{max_retries}: {e}")

            # 마지막 시도에서는 기다리지 않고 종료
            if attempt < max_retries:
                wait_seconds = 2 ** attempt
                print(f"   {wait_seconds}초 후 재시도합니다.")
                time.sleep(wait_seconds)

    raise last_error

In [12]:
log_lock = threading.Lock()


def save_json_result(video_id: str, result: dict) -> None:
    save_path = JSON_DIR / f"{video_id}.json"

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


def append_log(row: dict) -> None:
    log_df = pd.DataFrame([row])

    with log_lock:
        if LOG_PATH.exists():
            log_df.to_csv(LOG_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
        else:
            log_df.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")


def flatten_list_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in ["main_objects", "dominant_colors"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: "|".join(x) if isinstance(x, list) else x
            )

    return df

In [13]:
def get_processed_video_ids() -> set:
    processed = set()

    for json_file in JSON_DIR.glob("*.json"):
        processed.add(json_file.stem)

    return processed

In [ ]:
# 처리 완료 video_id 확인 함수 테스트
processed_ids = get_processed_video_ids()
print("이미 처리된 JSON 수:", len(processed_ids))

이미 처리된 JSON 수: 6


In [15]:
def process_one_video(row: pd.Series) -> dict:
    video_id = str(row["video_id"]).strip()

    channel_id = row["channel_id"] if "channel_id" in row.index else ""
    title = row["title"] if "title" in row.index else ""

    thumbnail_url = build_thumbnail_url(video_id)

    for attempt in range(1, PROCESS_MAX_RETRIES + 1):
        processed_at = datetime.now().isoformat(timespec="seconds")

        try:
            total_start = time.time()

            download_start = time.time()
            thumbnail_path = download_thumbnail(video_id, thumbnail_url)
            download_seconds = round(time.time() - download_start, 2)

            opencv_start = time.time()
            opencv_features = extract_opencv_features(thumbnail_path)
            opencv_seconds = round(time.time() - opencv_start, 2)

            gemini_start = time.time()
            gemini_features = analyze_thumbnail_with_gemini(thumbnail_path)
            gemini_seconds = round(time.time() - gemini_start, 2)

            total_seconds = round(time.time() - total_start, 2)

            result = {
                "video_id": video_id,
                "channel_id": channel_id,
                "title": title,
                "thumbnail_url": thumbnail_url,
                "thumbnail_path": thumbnail_path,

                **opencv_features,
                **gemini_features,

                "download_seconds": download_seconds,
                "opencv_seconds": opencv_seconds,
                "gemini_seconds": gemini_seconds,
                "total_seconds": total_seconds,

                "analysis_engine": "gemini",
                "analysis_model": MODEL_NAME,
                "prompt_version": PROMPT_VERSION,
                "schema_version": SCHEMA_VERSION,
                "status": "success",
                "error_message": None,
                "processed_at": processed_at,
            }

            save_json_result(video_id, result)

            append_log({
                "video_id": video_id,
                "status": "success",
                "stage": "done",
                "attempt": attempt,
                "error_message": "",
                "processed_at": processed_at,
            })

            return result

        except Exception as e:
            error_message = str(e)

            append_log({
                "video_id": video_id,
                "status": "retrying" if attempt < PROCESS_MAX_RETRIES else "failed",
                "stage": "process_one_video",
                "attempt": attempt,
                "error_message": error_message,
                "processed_at": datetime.now().isoformat(timespec="seconds"),
            })

            print(f"❌ 오류 발생: {video_id} / attempt={attempt} / {error_message}")

            if attempt < PROCESS_MAX_RETRIES:
                wait_seconds = 2 ** attempt
                print(f"   {wait_seconds}초 후 process_one_video 재시도")
                time.sleep(wait_seconds)
            else:
                result = {
                    "video_id": video_id,
                    "channel_id": channel_id,
                    "title": title,
                    "thumbnail_url": thumbnail_url,
                    "thumbnail_path": None,
                    "analysis_engine": "gemini",
                    "analysis_model": MODEL_NAME,
                    "prompt_version": PROMPT_VERSION,
                    "schema_version": SCHEMA_VERSION,
                    "status": "failed",
                    "error_message": error_message,
                    "processed_at": datetime.now().isoformat(timespec="seconds"),
                }

                save_json_result(video_id, result)
                return result

In [16]:
def rebuild_csv_from_json() -> pd.DataFrame:
    rows = []

    for json_file in JSON_DIR.glob("*.json"):
        with open(json_file, "r", encoding="utf-8") as f:
            row = json.load(f)
        rows.append(row)

    if len(rows) == 0:
        print("❌ JSON 파일이 없습니다.")
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = flatten_list_columns(df)

    if "video_id" in df.columns:
        df["video_id"] = df["video_id"].astype(str).str.strip()
        df = df.drop_duplicates(subset=["video_id"], keep="last")

    df.to_csv(CSV_OUTPUT_PATH, index=False, encoding="utf-8-sig")

    try:
        df.to_parquet(PARQUET_OUTPUT_PATH, index=False)
    except Exception as e:
        print(f"[WARN] parquet 저장 실패: {e}")

    print("✅ JSON 기준으로 CSV 재생성 완료")
    print("JSON 파일 수:", len(rows))
    print("CSV 행 수:", len(df))
    print("CSV 저장 경로:", CSV_OUTPUT_PATH)

    return df

In [17]:
# 단일 영상 테스트
test_result = process_one_video(input_df.iloc[0])
test_result

⚠️ Gemini 호출 실패 attempt=1/5: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
   2초 후 재시도합니다.


{'video_id': 'knCQBlBKSRM',
 'channel_id': 'UCwR9k7QggFEfeHkvTR31JcA',
 'title': '[B 다이렉트샵] 필사즉생의 각오로 임하는 발음 연습',
 'thumbnail_url': 'https://img.youtube.com/vi/knCQBlBKSRM/hqdefault.jpg',
 'thumbnail_path': 'c:\\Users\\user\\Desktop\\Uk_folder\\sparta_bootcamp\\project_collection\\final_project\\final_project\\works\\Hyeong_Uk\\thumb_image\\data\\thumbnails\\raw\\knCQBlBKSRM.jpg',
 'image_width': 480,
 'image_height': 360,
 'aspect_ratio': '4:3',
 'brightness_mean': 38.2,
 'contrast_std': 43.13,
 'saturation_mean': 111.33,
 'visual_summary': "초록색 식물 배경 앞에 한 남자가 돌 조각상 옆에 누워있고, 상단에는 '필사즉생의 각오로 임하는 홍철, 뭐 때문에?'라는 문구가 적혀 있습니다.",
 'main_objects': ['person', 'text', 'sculpture', 'plants'],
 'thumbnail_category': '인터뷰/인물형',
 'has_person': True,
 'person_count': 1,
 'has_face': True,
 'facial_expression': 'surprised',
 'has_text': True,
 'text_on_thumbnail': '필사즉생의 각오로 임하는 홍철, 뭐 때문에? 딴날',
 'text_language': 'ko',
 'text_size_level': 'large',
 'brand_logo_visible': False,
 'brand_name_visible': F

In [18]:
# json으로 csv 재성성 테스트
thumbnail_df = rebuild_csv_from_json()
thumbnail_df.head()

✅ JSON 기준으로 CSV 재생성 완료
JSON 파일 수: 6
CSV 행 수: 6
CSV 저장 경로: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image\data\thumbnails\thumbnail_features.csv


,video_id,channel_id,title,thumbnail_url,thumbnail_path,image_width,image_height,aspect_ratio,brightness_mean,contrast_std,...,opencv_seconds,gemini_seconds,total_seconds,analysis_engine,analysis_model,prompt_version,schema_version,status,error_message,processed_at
0,knCQBlBKSRM,UCwR9k7QggFEfeHkvTR31JcA,[B 다이렉트샵] 필사즉생의 각오로 임하는 발음 연습,https://img.youtube.com/vi/knCQBlBKSRM/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480,360,4:3,38.20,43.13,...,0.02,15.71,15.73,gemini,gemini-2.5-flash,v1,v1,success,None,2026-04-30T11:30:26
1,LRmLOsmMHts,UCwR9k7QggFEfeHkvTR31JcA,에이닷과 대화하며 즐기는 B tv IPTV로 바꾸자! (문의 1600-2345),https://img.youtube.com/vi/LRmLOsmMHts/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480,360,4:3,104.34,95.12,...,NaN,NaN,NaN,gemini,gemini-2.5-flash,v1,v1,success,None,2026-04-29T19:33:51
2,MIJR3Dm0YOk,UCsM07dUwo0WOWhFbVsG8K6A,학부모 참관수업에서 당황한 이유 | CU콘서트 #4강 #B조 #3라운드,https://img.youtube.com/vi/MIJR3Dm0YOk/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480,360,4:3,17.09,37.71,...,NaN,NaN,NaN,gemini,gemini-2.5-flash,v1,v1,success,None,2026-04-29T19:34:29
3,NCVxpXxTMSU,UCwR9k7QggFEfeHkvTR31JcA,[B 다이렉트샵] 고음불가? NO! 발음불가? YES,https://img.youtube.com/vi/NCVxpXxTMSU/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480,360,4:3,45.45,52.26,...,NaN,NaN,NaN,gemini,gemini-2.5-flash,v1,v1,success,None,2026-04-29T19:33:15
4,twi9zUxsXu0,UCsM07dUwo0WOWhFbVsG8K6A,🍓 + 🍫 +🌽 + 🥘 = 😋 | CU X 프롬서희 이모티콘 먹방 2월 3주차 #...,https://img.youtube.com/vi/twi9zUxsXu0/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480,360,4:3,83.47,66.09,...,NaN,NaN,NaN,gemini,gemini-2.5-flash,v1,v1,success,None,2026-04-29T19:34:02


In [19]:
def run_thumbnail_pipeline_sequential(
    input_df: pd.DataFrame,
    limit: Optional[int] = 10,
    skip_processed: bool = True,
) -> pd.DataFrame:
    run_df = input_df.copy()

    if limit is not None:
        run_df = run_df.head(limit).copy()

    run_df["video_id"] = run_df["video_id"].astype(str).str.strip()

    if skip_processed:
        processed_ids = get_processed_video_ids()
        before_count = len(run_df)

        run_df = run_df[~run_df["video_id"].isin(processed_ids)].copy()

        after_count = len(run_df)
        print(f"이미 처리된 video_id {before_count - after_count}개 skip")

    total_count = len(run_df)

    if total_count == 0:
        print("새로 처리할 video_id가 없습니다.")
        return rebuild_csv_from_json()

    print(f"🚀 순차 분석 시작: {total_count}개")

    for n, (_, row) in enumerate(run_df.iterrows(), start=1):
        video_id = str(row["video_id"]).strip()
        print(f"[{n}/{total_count}] 분석 중: {video_id}")
        process_one_video(row)

    print("✅ 순차 분석 완료")
    return rebuild_csv_from_json()

In [20]:
def chunk_dataframe(df: pd.DataFrame, chunk_size: int):
    for start in range(0, len(df), chunk_size):
        yield df.iloc[start:start + chunk_size]


def run_thumbnail_pipeline_parallel(
    input_df: pd.DataFrame,
    limit: Optional[int] = 10,
    skip_processed: bool = True,
    max_workers: int = DEFAULT_MAX_WORKERS,
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    chunk_sleep_seconds: int = DEFAULT_CHUNK_SLEEP_SECONDS,
) -> pd.DataFrame:
    run_df = input_df.copy()

    if limit is not None:
        run_df = run_df.head(limit).copy()

    run_df["video_id"] = run_df["video_id"].astype(str).str.strip()

    if skip_processed:
        processed_ids = get_processed_video_ids()
        before_count = len(run_df)

        run_df = run_df[~run_df["video_id"].isin(processed_ids)].copy()

        after_count = len(run_df)
        print(f"이미 처리된 video_id {before_count - after_count}개 skip")
    else:
        print("skip_processed=False: 이미 처리된 video_id도 다시 분석합니다.")

    total_count = len(run_df)

    if total_count == 0:
        print("새로 처리할 video_id가 없습니다.")
        return rebuild_csv_from_json()

    print("🚀 병렬 썸네일 분석 시작")
    print(f"총 처리 대상: {total_count}개")
    print(f"max_workers: {max_workers}")
    print(f"chunk_size: {chunk_size}")
    print(f"chunk_sleep_seconds: {chunk_sleep_seconds}")

    completed_count = 0

    for chunk_idx, chunk_df in enumerate(chunk_dataframe(run_df, chunk_size), start=1):
        print(f"\n===== Chunk {chunk_idx} 시작 / chunk 크기: {len(chunk_df)} =====")

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_video_id = {
                executor.submit(process_one_video, row): str(row["video_id"]).strip()
                for _, row in chunk_df.iterrows()
            }

            for future in as_completed(future_to_video_id):
                video_id = future_to_video_id[future]

                try:
                    result = future.result()
                    status = result.get("status", "unknown")

                    completed_count += 1
                    print(f"[{completed_count}/{total_count}] 완료: {video_id} / status={status}")

                except Exception as e:
                    completed_count += 1
                    print(f"[{completed_count}/{total_count}] ❌ 최종 실패: {video_id} / {e}")

                    fail_result = {
                        "video_id": video_id,
                        "status": "failed",
                        "error_message": str(e),
                        "processed_at": datetime.now().isoformat(timespec="seconds"),
                    }

                    save_json_result(video_id, fail_result)

        print(f"===== Chunk {chunk_idx} 종료 =====")
        print("현재까지 저장된 JSON 기준으로 CSV를 재생성합니다.")

        rebuild_csv_from_json()

        if chunk_sleep_seconds > 0:
            print(f"{chunk_sleep_seconds}초 대기 후 다음 chunk 진행")
            time.sleep(chunk_sleep_seconds)

    print("\n✅ 병렬 분석 전체 완료")
    return rebuild_csv_from_json()

In [21]:
start_time = time.time()

thumbnail_df_parallel_test = run_thumbnail_pipeline_parallel(
    input_df=input_df,
    limit=6,
    skip_processed=False,
    max_workers=2,
    chunk_size=3,
    chunk_sleep_seconds=3,
)

end_time = time.time()

elapsed = end_time - start_time

print(f"\n총 소요 시간: {elapsed:.2f}초")
print(f"분석 결과 행 수: {len(thumbnail_df_parallel_test)}")

display(thumbnail_df_parallel_test.head())

skip_processed=False: 이미 처리된 video_id도 다시 분석합니다.
🚀 병렬 썸네일 분석 시작
총 처리 대상: 6개
max_workers: 2
chunk_size: 3
chunk_sleep_seconds: 3

===== Chunk 1 시작 / chunk 크기: 3 =====
[1/6] 완료: knCQBlBKSRM / status=success
[2/6] 완료: NCVxpXxTMSU / status=success
[3/6] 완료: LRmLOsmMHts / status=success
===== Chunk 1 종료 =====
현재까지 저장된 JSON 기준으로 CSV를 재생성합니다.
✅ JSON 기준으로 CSV 재생성 완료
JSON 파일 수: 6
CSV 행 수: 6
CSV 저장 경로: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image\data\thumbnails\thumbnail_features.csv
3초 대기 후 다음 chunk 진행

===== Chunk 2 시작 / chunk 크기: 3 =====
⚠️ Gemini 호출 실패 attempt=1/5: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googl

,video_id,channel_id,title,thumbnail_url,thumbnail_path,image_width,image_height,aspect_ratio,brightness_mean,contrast_std,...,opencv_seconds,gemini_seconds,total_seconds,analysis_engine,analysis_model,prompt_version,schema_version,status,error_message,processed_at
0,knCQBlBKSRM,UCwR9k7QggFEfeHkvTR31JcA,[B 다이렉트샵] 필사즉생의 각오로 임하는 발음 연습,https://img.youtube.com/vi/knCQBlBKSRM/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480.0,360.0,4:3,38.20,43.13,...,0.00,10.00,10.01,gemini,gemini-2.5-flash,v1,v1,success,NaN,2026-04-30T11:36:26
1,LRmLOsmMHts,UCwR9k7QggFEfeHkvTR31JcA,에이닷과 대화하며 즐기는 B tv IPTV로 바꾸자! (문의 1600-2345),https://img.youtube.com/vi/LRmLOsmMHts/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480.0,360.0,4:3,104.34,95.12,...,0.00,9.71,9.71,gemini,gemini-2.5-flash,v1,v1,success,NaN,2026-04-30T11:36:36
2,MIJR3Dm0YOk,UCsM07dUwo0WOWhFbVsG8K6A,학부모 참관수업에서 당황한 이유 | CU콘서트 #4강 #B조 #3라운드,https://img.youtube.com/vi/MIJR3Dm0YOk/hqdefau...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,gemini,gemini-2.5-flash,v1,v1,failed,429 RESOURCE_EXHAUSTED. {'error': {'code': 429...,2026-04-30T11:37:29
3,NCVxpXxTMSU,UCwR9k7QggFEfeHkvTR31JcA,[B 다이렉트샵] 고음불가? NO! 발음불가? YES,https://img.youtube.com/vi/NCVxpXxTMSU/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480.0,360.0,4:3,45.45,52.26,...,0.01,10.65,10.66,gemini,gemini-2.5-flash,v1,v1,success,NaN,2026-04-30T11:36:26
4,twi9zUxsXu0,UCsM07dUwo0WOWhFbVsG8K6A,🍓 + 🍫 +🌽 + 🥘 = 😋 | CU X 프롬서희 이모티콘 먹방 2월 3주차 #...,https://img.youtube.com/vi/twi9zUxsXu0/hqdefau...,c:\Users\user\Desktop\Uk_folder\sparta_bootcam...,480.0,360.0,4:3,83.47,66.09,...,0.01,8.32,8.33,gemini,gemini-2.5-flash,v1,v1,success,NaN,2026-04-30T11:36:49


# 셀 18. 100개 테스트용

In [ ]:
# start_time = time.time()

# thumbnail_df_100 = run_thumbnail_pipeline_parallel(
#     input_df=input_df,
#     limit=100,
#     skip_processed=True,
#     max_workers=2,
#     chunk_size=25,
#     chunk_sleep_seconds=5,
# )

# end_time = time.time()

# elapsed = end_time - start_time
# processed_count = min(100, len(input_df))

# print(f"\n총 소요 시간: {elapsed:.2f}초")
# print(f"1개당 평균 시간: {elapsed / processed_count:.2f}초")
# print(f"20,000개 예상 시간: {(elapsed / processed_count) * 20000 / 3600:.2f}시간")

# 셀 19. 결과 확인

In [22]:
thumbnail_df = rebuild_csv_from_json()

check_cols = [
    "video_id",
    "thumbnail_category",
    "has_person",
    "person_count",
    "has_text",
    "text_on_thumbnail",
    "brand_logo_visible",
    "brand_name_detected",
    "clickbait_level",
    "professional_level",
    "brightness_mean",
    "contrast_std",
    "saturation_mean",
    "download_seconds",
    "opencv_seconds",
    "gemini_seconds",
    "total_seconds",
    "status",
]

existing_cols = [col for col in check_cols if col in thumbnail_df.columns]

print("데이터 크기:", thumbnail_df.shape)
print("확인 가능한 컬럼:", existing_cols)

display(thumbnail_df[existing_cols].head(20))

✅ JSON 기준으로 CSV 재생성 완료
JSON 파일 수: 6
CSV 행 수: 6
CSV 저장 경로: c:\Users\user\Desktop\Uk_folder\sparta_bootcamp\project_collection\final_project\final_project\works\Hyeong_Uk\thumb_image\data\thumbnails\thumbnail_features.csv
데이터 크기: (6, 44)
확인 가능한 컬럼: ['video_id', 'thumbnail_category', 'has_person', 'person_count', 'has_text', 'text_on_thumbnail', 'brand_logo_visible', 'brand_name_detected', 'clickbait_level', 'professional_level', 'brightness_mean', 'contrast_std', 'saturation_mean', 'download_seconds', 'opencv_seconds', 'gemini_seconds', 'total_seconds', 'status']


,video_id,thumbnail_category,has_person,person_count,has_text,text_on_thumbnail,brand_logo_visible,brand_name_detected,clickbait_level,professional_level,brightness_mean,contrast_std,saturation_mean,download_seconds,opencv_seconds,gemini_seconds,total_seconds,status
0,knCQBlBKSRM,인터뷰/인물형,True,1.0,True,"필사즉생의 각오로 임하는 홍철, 뭐 때문에? 맘마살",False,,3.0,4.0,38.20,43.13,111.33,0.0,0.00,10.00,10.01,success
1,LRmLOsmMHts,제품홍보형,True,1.0,True,에이닷과 대화하며 즐기는 Btv로 바꾸자특별한 혜택까지!,True,Btv,2.0,4.0,104.34,95.12,55.46,0.0,0.00,9.71,9.71,success
2,MIJR3Dm0YOk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,failed
3,NCVxpXxTMSU,정보전달형,True,1.0,True,노홍철은 고음불가? NO! 발음불가? YES!! B나이다 흥정마시어~,False,,3.0,3.0,45.45,52.26,75.42,0.0,0.01,10.65,10.66,success
4,twi9zUxsXu0,제품홍보형,True,1.0,True,HEYROO POPCORN 대파콘소메맛 팝콘,True,HEYROO,1.0,3.0,83.47,66.09,76.01,0.0,0.01,8.32,8.33,success
5,Xfu1kCID0Ls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,failed


# 셀 20. 처리 시간 요약

In [23]:
time_cols = [
    "download_seconds",
    "opencv_seconds",
    "gemini_seconds",
    "total_seconds",
]

existing_time_cols = [col for col in time_cols if col in thumbnail_df.columns]

if existing_time_cols:
    display(thumbnail_df[existing_time_cols].describe())
else:
    print("처리 시간 컬럼이 없습니다.")

,download_seconds,opencv_seconds,gemini_seconds,total_seconds
count,4.0,4.000000,4.000000,4.00000
mean,0.0,0.005000,9.670000,9.67750
std,0.0,0.005774,0.982073,0.98195
min,0.0,0.000000,8.320000,8.33000
25%,0.0,0.000000,9.362500,9.36500
50%,0.0,0.005000,9.855000,9.86000
75%,0.0,0.010000,10.162500,10.17250
max,0.0,0.010000,10.650000,10.66000


# 셀 21. 상태 및 분류 결과 확인

In [24]:
if "status" in thumbnail_df.columns:
    print("[status 분포]")
    display(thumbnail_df["status"].value_counts())

if "thumbnail_category" in thumbnail_df.columns:
    print("[thumbnail_category 분포]")
    display(thumbnail_df["thumbnail_category"].value_counts())

if "clickbait_level" in thumbnail_df.columns:
    print("[clickbait_level 분포]")
    display(thumbnail_df["clickbait_level"].value_counts().sort_index())

if "professional_level" in thumbnail_df.columns:
    print("[professional_level 분포]")
    display(thumbnail_df["professional_level"].value_counts().sort_index())

[status 분포]


status
success    4
failed     2
Name: count, dtype: int64

[thumbnail_category 분포]


thumbnail_category
제품홍보형      2
인터뷰/인물형    1
정보전달형      1
Name: count, dtype: int64

[clickbait_level 분포]


clickbait_level
1.0    1
2.0    1
3.0    2
Name: count, dtype: int64

[professional_level 분포]


professional_level
3.0    2
4.0    2
Name: count, dtype: int64

# 셀 22. 전체 실행용

In [ ]:
# thumbnail_df_all = run_thumbnail_pipeline_parallel(
#     input_df=input_df,
#     limit=None,
#     skip_processed=True,
#     max_workers=2,
#     chunk_size=50,
#     chunk_sleep_seconds=5,
# )

# display(thumbnail_df_all.head())